# VARNN Forecast Analysis

Notebook tự động tạo để tối ưu lag, theo dõi train/validation và biểu đồ test.

In [1]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from statsmodels.tsa.api import VAR
from statsmodels.tsa.statespace.sarimax import SARIMAX

warnings.filterwarnings("ignore")

TARGET = "cpi_mom"
DATA_PATH = "features_with_gdp.csv"

TRAIN_SIZE = 0.70
VAL_SIZE = 0.15
LAG_GRID = [1, 3, 6, 9, 12]
SARIMAX_ORDERS = [(1, 0, 0), (1, 0, 1), (2, 0, 1)]

EPOCHS = 300
LR = 1e-3
BATCH_SIZE = 32
SEED = 42

np.random.seed(SEED)
torch.manual_seed(SEED)


def metric(y_true, y_pred):
    y_true = pd.Series(y_true).dropna()
    y_pred = pd.Series(y_pred).dropna()
    idx = y_true.index.intersection(y_pred.index)
    if len(idx) == 0:
        return {"rmse": np.nan, "mae": np.nan, "r2": np.nan}
    y_true, y_pred = y_true.loc[idx], y_pred.loc[idx]
    return {
        "rmse": np.sqrt(mean_squared_error(y_true, y_pred)),
        "mae": mean_absolute_error(y_true, y_pred),
        "r2": r2_score(y_true, y_pred)
    }


def load_data(target):
    if "features_gdp" in globals():
        df = features_gdp.copy()
    else:
        df = pd.read_csv(DATA_PATH)

    df["date"] = pd.to_datetime(df["date"])
    df = df.sort_values("date").set_index("date")
    df = df.loc[:, ~df.columns.duplicated()]
    df = df.select_dtypes(include=[np.number])
    df = df.replace([np.inf, -np.inf], np.nan).ffill().dropna()

    if "policy_rate.1" in df.columns:
        df = df.drop(columns=["policy_rate.1"])

    cols = [
        target,
        "policy_rate", "broad_money", "ppi_qoq", "wti", "gasoline_world",
        "gold", "VNINDEX", "NIKKEI225", "USDVND", "gdp",
        "gdp_cong_nghiep_va_xay_dung_gia_tri_hien_hanh",
        "gdp_dich_vu_gia_tri_hien_hanh",
        "gdp_nong_lam_nghiep_va_thuy_san_gia_tri_hien_hanh"
    ]
    cols = [c for c in cols if c in df.columns]
    return df[cols].copy()


def split_train_val_test(df):
    n = len(df)
    train_end = int(n * TRAIN_SIZE)
    val_end = int(n * (TRAIN_SIZE + VAL_SIZE))
    train = df.iloc[:train_end].copy()
    val = df.iloc[train_end:val_end].copy()
    test = df.iloc[val_end:].copy()
    train_val = df.iloc[:val_end].copy()
    return train, val, test, train_val


def make_ts_data(df, target, lag):
    values = df.values
    target_idx = df.columns.get_loc(target)
    X, y, idx = [], [], []
    for i in range(lag, len(df)):
        X.append(values[i-lag:i])
        y.append(values[i, target_idx])
        idx.append(df.index[i])
    return np.array(X), np.array(y), pd.Index(idx)


def make_exog_lag(X, lag):
    out = []
    for l in range(1, lag + 1):
        temp = X.shift(l)
        temp.columns = [f"{c}_lag{l}" for c in X.columns]
        out.append(temp)
    return pd.concat(out, axis=1).dropna()


def plot_test(actual, pred, title):
    m = metric(actual, pred)
    plt.figure(figsize=(14, 5))
    plt.plot(actual.index, actual.values, label="Actual", linewidth=2)
    plt.plot(pred.index, pred.values, label="Forecast", linewidth=2)
    plt.title(f"{title} | RMSE={m['rmse']:.4f}, MAE={m['mae']:.4f}, R2={m['r2']:.4f}")
    plt.xlabel("Date")
    plt.ylabel(TARGET)
    plt.legend()
    plt.tight_layout()
    plt.show()


def plot_train_val_loss(history, title):
    plt.figure(figsize=(10, 4))
    plt.plot(history["train_loss"], label="Train Loss", linewidth=2)
    plt.plot(history["val_loss"], label="Validation Loss", linewidth=2)
    plt.title(title)
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.tight_layout()
    plt.show()


In [2]:
class VARNN(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64), nn.ReLU(), nn.Dropout(0.10),
            nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, 1)
        )
    def forward(self, x):
        return self.net(x.reshape(x.size(0), -1)).squeeze(-1)

class GRUModel(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.gru = nn.GRU(n_features, 64, batch_first=True)
        self.fc = nn.Linear(64, 1)
    def forward(self, x):
        out, _ = self.gru(x)
        return self.fc(out[:, -1, :]).squeeze(-1)

class LSTMModel(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.lstm = nn.LSTM(n_features, 64, batch_first=True)
        self.fc = nn.Sequential(nn.Linear(64, 32), nn.ReLU(), nn.Dropout(0.10), nn.Linear(32, 1))
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :]).squeeze(-1)

class TCNBlock(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size=2, dilation=1, dropout=0.10):
        super().__init__()
        padding = (kernel_size - 1) * dilation
        self.conv = nn.Conv1d(in_ch, out_ch, kernel_size=kernel_size, dilation=dilation, padding=padding)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
        self.downsample = nn.Conv1d(in_ch, out_ch, kernel_size=1) if in_ch != out_ch else None
    def forward(self, x):
        out = self.conv(x)
        out = out[:, :, :x.size(2)]
        out = self.dropout(self.relu(out))
        residual = x if self.downsample is None else self.downsample(x)
        return self.relu(out + residual)

class TCNModel(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.tcn = nn.Sequential(
            TCNBlock(n_features, 32, dilation=1),
            TCNBlock(32, 32, dilation=2),
            TCNBlock(32, 32, dilation=4)
        )
        self.fc = nn.Linear(32, 1)
    def forward(self, x):
        x = x.transpose(1, 2)
        out = self.tcn(x)
        return self.fc(out[:, :, -1]).squeeze(-1)


def get_nn_model(model_type, n_features, lag):
    if model_type == "VARNN": return VARNN(lag * n_features)
    if model_type == "GRU": return GRUModel(n_features)
    if model_type == "LSTM": return LSTMModel(n_features)
    if model_type == "TCN": return TCNModel(n_features)
    raise ValueError(model_type)


def fit_predict_nn(fit_df, val_df, full_df, eval_index, target, lag, model_type, keep_history=False):
    scaler = StandardScaler()
    fit_s = pd.DataFrame(scaler.fit_transform(fit_df), index=fit_df.index, columns=fit_df.columns)
    full_s = pd.DataFrame(scaler.transform(full_df), index=full_df.index, columns=full_df.columns)

    X_all, y_all, idx_all = make_ts_data(full_s, target, lag)
    train_mask = idx_all.isin(fit_df.index)
    val_mask = idx_all.isin(val_df.index)
    eval_mask = idx_all.isin(eval_index)

    X_train = torch.tensor(X_all[train_mask], dtype=torch.float32)
    y_train = torch.tensor(y_all[train_mask], dtype=torch.float32)
    X_val = torch.tensor(X_all[val_mask], dtype=torch.float32)
    y_val = torch.tensor(y_all[val_mask], dtype=torch.float32)
    X_eval = torch.tensor(X_all[eval_mask], dtype=torch.float32)
    idx_eval = idx_all[eval_mask]

    model = get_nn_model(model_type, full_df.shape[1], lag)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
    loss_fn = nn.SmoothL1Loss()
    loader = torch.utils.data.DataLoader(
        torch.utils.data.TensorDataset(X_train, y_train),
        batch_size=min(BATCH_SIZE, len(X_train)), shuffle=False
    )

    history = {"train_loss": [], "val_loss": []}
    for _ in range(EPOCHS):
        model.train()
        batch_losses = []
        for xb, yb in loader:
            optimizer.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            optimizer.step()
            batch_losses.append(loss.item())
        model.eval()
        with torch.no_grad():
            val_loss = loss_fn(model(X_val), y_val).item() if len(X_val) else np.nan
        history["train_loss"].append(float(np.mean(batch_losses)))
        history["val_loss"].append(val_loss)

    model.eval()
    with torch.no_grad():
        pred_s = model(X_eval).numpy()

    target_idx = full_df.columns.get_loc(target)
    dummy = np.zeros((len(pred_s), full_df.shape[1]))
    dummy[:, target_idx] = pred_s
    pred = scaler.inverse_transform(dummy)[:, target_idx]
    pred = pd.Series(pred, index=idx_eval, name=model_type)
    return pred, history


def optimize_nn(df, target, model_type):
    train, val, test, train_val = split_train_val_test(df)
    rows = []
    for lag in LAG_GRID:
        try:
            pred_val, hist = fit_predict_nn(train, val, df, val.index, target, lag, model_type)
            m = metric(val[target], pred_val)
            rows.append({"model": model_type, "lag": lag, "val_rmse": m["rmse"], "history": hist})
        except Exception as e:
            rows.append({"model": model_type, "lag": lag, "val_rmse": np.nan, "history": None})
    tune_df = pd.DataFrame([{k:v for k,v in r.items() if k != "history"} for r in rows]).sort_values("val_rmse")
    best = next(r for r in rows if r["lag"] == int(tune_df.iloc[0]["lag"]))
    pred_test, final_hist = fit_predict_nn(train_val, val, df, test.index, target, best["lag"], model_type)
    return tune_df, best, pred_test, final_hist


In [3]:
df = load_data(TARGET)
train, val, test, train_val = split_train_val_test(df)
tune_df, best, pred_test, history = optimize_nn(df, TARGET, "VARNN")
test_y = test[TARGET]
result_df = pd.DataFrame([{{"model": "VARNN", "best_lag": best["lag"], "val_rmse": best["val_rmse"], **metric(test_y, pred_test)}}])
print("Validation tuning")
display(tune_df)
print("Test result")
display(result_df)
plot_train_val_loss(history, "VARNN train loss and validation loss")
plot_test(test_y, pred_test, "VARNN test forecast")
forecast_df = pd.DataFrame({{"actual": test_y, "VARNN": pred_test}})
forecast_df.tail()


KeyError: 'date'